# 05 · Serie de tiempo: ¿se puede pronosticar el tráfico aéreo?
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/05_series_tiempo_trafico_aereo.ipynb)

**El dato:** pasajeros diarios que pasaron por los controles de seguridad de los aeropuertos de Estados Unidos (TSA), desde el 1 de enero de 2019: 2.766 días reales, con el colapso de la pandemia incluido.

**Familia:** supervisado. El pasado funciona como la respuesta conocida: el modelo aprende hasta diciembre de 2025 y debe pronosticar enero y febrero de 2026 sin verlos.

**Cómo funciona ARIMA estacional:** predice cada día con los días anteriores, sus errores recientes y lo que pasó el mismo día de la semana anterior (estacionalidad semanal).

In [ ]:
# Preparación: trae los datos del repositorio (funciona igual en Google Colab)
import os, urllib.request
REPO = "https://raw.githubusercontent.com/andrezconz/investigacion-aumentada-ml/main/datos/"

def traer(nombre):
    local = os.path.join("..", "datos", nombre)
    if os.path.exists(local):
        return local
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(REPO + nombre, nombre)
    return nombre

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv(traer("tsa_trafico_diario.csv"), parse_dates=["fecha"]).sort_values("fecha").reset_index(drop=True)
print(len(df), "días ·", df.fecha.min().date(), "a", df.fecha.max().date())
df.tail()

In [ ]:
import matplotlib.pyplot as plt
minimo = df.loc[df["pasajeros"].idxmin()]
prom_2019 = df.loc[df["fecha"] < "2020-01-01", "pasajeros"].mean()
print(f"Mínimo: {minimo.fecha.date()} con {minimo.pasajeros:,.0f} pasajeros ({(1 - minimo.pasajeros / prom_2019) * 100:.1f} % menos que el promedio de 2019)")

plt.figure(figsize=(14, 4.5))
plt.plot(df["fecha"], df["pasajeros"], color="#020E60", lw=.5)
plt.scatter([minimo.fecha], [minimo.pasajeros], color="#C0392B", zorder=5)
plt.title("Pasajeros diarios en controles de la TSA"); plt.ylabel("Pasajeros por día"); plt.show()

In [ ]:
# Patrón semanal (2022 en adelante, para no mezclar el choque de la pandemia)
dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
nombres = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
sem = df[df["fecha"] >= "2022-01-01"].groupby(df["fecha"].dt.day_name())["pasajeros"].mean().reindex(dias)
sem.index = nombres
sem.round(0)

## Entrenar hasta 2025 y pronosticar 59 días
Comparamos contra una alternativa **ingenua estacional**: repetir lo que pasó el mismo día de la semana anterior. Si el modelo no le gana a eso, no sirve.

Se entrena desde 2023 para dejar por fuera el choque de la pandemia.

In [ ]:
import warnings
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")

corte, horizonte = pd.Timestamp("2025-12-31"), 59
ent = df[(df["fecha"] >= "2023-01-01") & (df["fecha"] <= corte)]
prueba = df[(df["fecha"] > corte)].head(horizonte)

modelo = ARIMA(ent["pasajeros"].values, order=(3, 0, 2), seasonal_order=(0, 1, 1, 7)).fit()
pron = modelo.forecast(horizonte)
ingenuo = np.resize(ent["pasajeros"].values[-7:], horizonte)

rmse = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
print(f"Error del modelo (RMSE):          {rmse(prueba.pasajeros, pron):,.0f} pasajeros por día")
print(f"Error del ingenuo estacional:     {rmse(prueba.pasajeros, ingenuo):,.0f} pasajeros por día")

In [ ]:
reciente = df[(df["fecha"] > corte - pd.Timedelta(days=60)) & (df["fecha"] <= corte)]
plt.figure(figsize=(13, 5))
plt.plot(reciente.fecha, reciente.pasajeros, color="#020E60", lw=1.4)
plt.plot(prueba.fecha, prueba.pasajeros, color="#020E60", lw=1.4, label="Lo que pasó")
plt.plot(prueba.fecha, pron, color="#2E5FAE", ls="--", lw=2, label="Pronóstico ARIMA")
plt.plot(prueba.fecha, ingenuo, color="#D4A017", ls=":", lw=2, label="Repetir la semana anterior")
plt.axvline(corte, color="grey", ls=":"); plt.legend(); plt.ylabel("Pasajeros por día")
plt.title("El modelo solo vio hasta el 31 de diciembre de 2025"); plt.show()

> **Tip:** compara siempre contra una alternativa simple, y mira la trayectoria completa, no solo el número de error: a inicios de febrero de 2026 hubo una caída puntual que ni el modelo ni el ingenuo anticiparon. Ningún modelo entrenado solo con el pasado anticipa un choque puntual.

*Fuente: TSA (Transportation Security Administration), volúmenes diarios de pasajeros publicados en tsa.gov. Elaboración propia. El orden del modelo, ARIMA(3,0,2)(0,1,1)[7], se escogió por criterio AIC.*